# NamedVertexCoarsener example

`NamedVertexCoarsener` is the deliberately simple schema-1 method: configure a
label or UID set, fit, transform, and decode. Shared machinery supplies exact
types, attachments, provenance, lineage, and validation.


In [ ]:
from tree_coarsening import NamedVertexCoarsener
from tree_coarsening.utils import make_named_component_tree

raw = make_named_component_tree(
    component_sizes=(5, 3),
    selected_labels=("A", "B"),
    include_singleton=True,
    seed=0,
)
print(f"raw tree: {raw.number_of_nodes()} nodes, {raw.number_of_edges()} edges")

In [ ]:
by_label = NamedVertexCoarsener(
    labels={"A", "B"},
    component_policy="all",
    model_id="named-label-example",
).fit([raw])
encoded_by_label = by_label.transform(raw)
print("label-selected nodes:", encoded_by_label.number_of_nodes())
print("vocabulary:", by_label.encoder_.vocab.as_dict())

In [ ]:
selected_uids = {
    data["uid"] for _, data in raw.nodes(data=True) if data["uid"].startswith("named_component_0_")
}
by_uid = NamedVertexCoarsener(
    uids=selected_uids,
    component_policy="all",
    model_id="named-uid-example",
).fit([raw])
encoded_by_uid = by_uid.transform(raw)
print("UID-selected nodes:", encoded_by_uid.number_of_nodes())

In [ ]:
for node, data in encoded_by_uid.nodes(data=True):
    if data["label"] in by_uid.encoder_.vocab:
        print(
            f"node={node} label={data['label']!r} size={data['size']} "
            f"super_uids={data['super_uids']!r}"
        )

In [ ]:
def raw_signature(graph):
    nodes = sorted(
        (data["uid"], tuple(sorted(data.items(), key=lambda item: repr(item[0]))))
        for _, data in graph.nodes(data=True)
    )
    edges = sorted(
        (graph.nodes[parent]["uid"], graph.nodes[child]["uid"]) for parent, child in graph.edges
    )
    return nodes, edges


for model, encoded in ((by_label, encoded_by_label), (by_uid, encoded_by_uid)):
    assert raw_signature(model.decode(encoded)) == raw_signature(raw)
print("all roundtrips ok")